In [1]:
import pyspark

In [2]:
from pyspark.sql import SparkSession
import os

# 1. Setup Credential
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/workspaces/zoomcamp-project/keys/gcp-key.json"

# 2. Inisialisasi Spark dengan FIX untuk Java 17
spark = SparkSession.builder \
    .appName('GCS-Connect-Java17-Fix') \
    .config("spark.jars.packages", "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.14") \
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS") \
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile", os.environ["GOOGLE_APPLICATION_CREDENTIALS"]) \
    .config("spark.driver.extraJavaOptions", "--add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED") \
    .config("spark.executor.extraJavaOptions", "--add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/18 16:39:27 WARN Utils: Your hostname, codespaces-59d644, resolves to a loopback address: 127.0.0.1; using 10.0.0.227 instead (on interface eth0)
26/03/18 16:39:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/codespace/pyspark-zoomcamp-venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/codespace/.ivy2.5.2/cache
The jars for the packages stored in: /home/codespace/.ivy2.5.2/jars
com.google.cloud.bigdataoss#gcs-connector added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ebfeb398-7a1e-4172-b051-13499b3f2656;1.0
	confs: [default]
	found com.google.cloud.bigdataoss#gcs-connector;hadoop3-2.2.14 in central
	found com.google.api-client#google-api-client-jackson2;2.0.1 in central
	found com.google.api-client#google-ap

Berhasil! Spark Session aktif di Java 17.
Spark Session Berhasil Dibuat!


In [5]:
from pyspark.sql import functions as F

# 2. Daftar Tabel Olist
tables = [
    "customers",
    "geolocation",
    "order_items",
    "payments",
    "reviews",
    "orders",
    "products",
    "sellers",
    "product_category_translation"
]

base_input = "gs://ammar-zoomcamp-project/raw"
base_output = "gs://ammar-zoomcamp-project/processed"

# 3. Looping Proses
for table in tables:
    try:
        print(f"🔄 Memproses {table}...")
        
        # Baca CSV (Lazy Load - Belum makan RAM)
        df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{base_input}/{table}.csv")
        
        df = df.dropna(how='all')

        # 2. Trim whitespace (menghapus spasi di awal/akhir teks)
        for col_name, col_type in df.dtypes:
            if col_type == "string":
                df = df.withColumn(col_name, F.trim(F.col(col_name)))


        # Langsung Tulis ke Parquet (Data mengalir langsung ke GCS)
        df.write.mode("overwrite").parquet(f"{base_output}/{table}")
        
        print(f"✅ {table} Berhasil dikonversi!")
        
        # Hapus variabel untuk mengosongkan RAM Python
        del df
        
    except Exception as e:
        print(f"❌ Gagal memproses {table}: {str(e)}")

print("\n🚀 SEMUA TABEL SELESAI. Anda bisa lanjut ke Kestra/BigQuery!")

🔄 Memproses customers...


✅ customers Berhasil dikonversi!
🔄 Memproses geolocation...


✅ geolocation Berhasil dikonversi!
🔄 Memproses order_items...


✅ order_items Berhasil dikonversi!
🔄 Memproses payments...


✅ payments Berhasil dikonversi!
🔄 Memproses reviews...


✅ reviews Berhasil dikonversi!
🔄 Memproses orders...


✅ orders Berhasil dikonversi!
🔄 Memproses products...


✅ products Berhasil dikonversi!
🔄 Memproses sellers...


✅ sellers Berhasil dikonversi!
🔄 Memproses product_category_translation...


✅ product_category_translation Berhasil dikonversi!

🚀 SEMUA TABEL SELESAI. Anda bisa lanjut ke Kestra/BigQuery!
